# BraTS2024 — MedRT-SFSeg source-free modality adaptation

Bốn hướng: T1n→T2w, T2w→T1n, T1c→T2f, T2f→T1c. Model chính YOLO26-S-Seg + AdaBN + Mean Teacher + Mask-DHF + class-wise DURR + SegMARD-v2. Target labels chỉ dùng sau khi chốt model để test. Xem `docs/BraTS2024.md`. Chọn runtime GPU trước khi train.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
import shutil, subprocess, sys

DRIVE_PROJECT = Path("/content/drive/MyDrive/MedRT-SFOD")
WORK = Path("/content/MedRT-SFOD")
WORK.mkdir(parents=True, exist_ok=True)
RAW_LOCAL = Path("/content/BraTS2024")
PREPARED = Path("/content/BraTS2024-SFSeg")
OUTPUT = DRIVE_PROJECT / "runs/seg/brats2024_sfseg"
DIRECTIONS = ["t1n_to_t2w", "t2w_to_t1n", "t1c_to_t2f", "t2f_to_t1c"]
RESUME = False  # True sau khi run bị ngắt; giữ nguyên cấu hình


In [ ]:
for name in ("scripts", "ultralytics", "configs", "docs"):
    shutil.copytree(DRIVE_PROJECT / name, WORK / name, dirs_exist_ok=True)
for name in ("pyproject.toml", "README.md", "requirements-brats.txt"):
    shutil.copy2(DRIVE_PROJECT / name, WORK / name)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(WORK), "-r", str(WORK / "requirements-brats.txt")], check=True)
import torch
assert torch.cuda.is_available(), "Chọn runtime GPU để huấn luyện đầy đủ."
print(torch.cuda.get_device_name(0))


Chuẩn bị MRI trên ổ local để tránh đọc/ghi hàng triệu file nhỏ trên Drive. Cell sau chỉ copy ba thư mục GLI và metadata. Nếu runtime bị reset, chạy lại chuẩn bị theo cùng seed để tái tạo đúng split; checkpoints vẫn ở Drive.

In [ ]:
for name in ("training_data1_v2", "training_data_additional", "validation_data"):
    print("Copy:", name, flush=True)
    shutil.copytree(DRIVE_PROJECT / "dataset/BraTS2024" / name, RAW_LOCAL / name, dirs_exist_ok=True)
metadata = "BraTS-PTG supplementary demographic information and metadata.xlsx"
shutil.copy2(DRIVE_PROJECT / "dataset/BraTS2024" / metadata, RAW_LOCAL / metadata)
RUNNER = WORK / "scripts/YOLO26/medseg/brats2024_experiments.py"
BASE = [sys.executable, str(RUNNER), "--raw-root", str(RAW_LOCAL), "--prepared-root", str(PREPARED), "--project", str(OUTPUT), "--device", "0"]
def run_stage(stage, direction=None):
    cmd = BASE + ["--stage", stage]
    if direction:
        cmd += ["--directions", direction]
    if RESUME:
        cmd += ["--resume"]
    subprocess.run(cmd, cwd=WORK, check=True)
run_stage("plan")


In [ ]:
run_stage("prepare")


Train source theo source val; adaptation chỉ đọc ảnh target train. Evaluation được chạy sau khi lưu final Student. Mỗi hướng lưu vào thư mục riêng; chạy lại sẽ bỏ qua stage hoàn tất và kiểm tra hash. Với T4, cấu hình mặc định 256×256, batch 8; nếu đổi batch, phải dùng cấu hình đó nhất quán giữa các stage và khi resume.

In [ ]:
for direction in DIRECTIONS:
    print("Experiment:", direction, flush=True)
    run_stage("all", direction)


In [ ]:
run_stage("summarize")
import pandas as pd
display(pd.read_csv(OUTPUT / "full/summary.csv"))


Tuỳ chọn: suy luận 188 ca validation chính thức không nhãn. Kết quả là NIfTI segmentation; không có Dice/ASD vì không có GT. Không dùng tập này để chọn checkpoint.

In [ ]:
# run_stage("predict", "t1n_to_t2w")
